<a href="https://colab.research.google.com/github/aSafarpoor/storehouse/blob/main/SLM_sample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install llama-cpp-python
!pip install llama-cpp-python --quiet # Prefered

In [ ]:
# Model
from llama_cpp import Llama

llm = Llama.from_pretrained(
	repo_id="NoelJacob/Meta-Llama-3-8B-Instruct-Q4_K_M-GGUF",
	filename="meta-llama-3-8b-instruct.Q4_K_M.gguf",   ##################Model Name We Are Using########################
)

./meta-llama-3-8b-instruct.Q4_K_M.gguf:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

llama_model_loader: loaded meta data with 21 key-value pairs and 291 tensors from /root/.cache/huggingface/hub/models--NoelJacob--Meta-Llama-3-8B-Instruct-Q4_K_M-GGUF/snapshots/a34df0cb86e31caf879f50e33d7da1d79dc4eb17/./meta-llama-3-8b-instruct.Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = Meta-Llama-3-8B-Instruct
llama_model_loader: - kv   2:                          llama.block_count u32              = 32
llama_model_loader: - kv   3:                       llama.context_length u32              = 8192
llama_model_loader: - kv   4:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_load

In [ ]:
llm.create_chat_completion(
	messages = [
		{
			"role": "user",
			"content": "What is the capital of France?" # Simple Sample Asking a Question
		}
	]
)

Llama.generate: 16 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =   11454.34 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =    6059.82 ms /     8 runs   (  757.48 ms per token,     1.32 tokens per second)
llama_perf_context_print:       total time =    6073.09 ms /     9 tokens
llama_perf_context_print:    graphs reused =          8


{'id': 'chatcmpl-9848a336-2518-4059-93ac-acfb77ca0aad',
 'object': 'chat.completion',
 'created': 1764044994,
 'model': '/root/.cache/huggingface/hub/models--NoelJacob--Meta-Llama-3-8B-Instruct-Q4_K_M-GGUF/snapshots/a34df0cb86e31caf879f50e33d7da1d79dc4eb17/./meta-llama-3-8b-instruct.Q4_K_M.gguf',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'The capital of France is Paris.'},
   'logprobs': None,
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 17, 'completion_tokens': 7, 'total_tokens': 24}}

In [ ]:
import time


# -------------------------
# Load your model # Redundant, but to have all code in same place.
# -------------------------
llm = Llama.from_pretrained(
    repo_id="NoelJacob/Meta-Llama-3-8B-Instruct-Q4_K_M-GGUF",
    filename="meta-llama-3-8b-instruct.Q4_K_M.gguf",
    n_ctx=4096,
    n_gpu_layers=-1,   # GPU-accelerated on Colab
    verbose=False
)


# -------------------------
# Define the 4 agents and Their Attributes, it is fine to define other Attributes, the JSON format is prefered for tokeniser
# -------------------------
agents = [
    {
        "name": "Marcus",
        "club": "Arsenal",
        "stubbornness": 90,
        "persona": """
Marcus is a DIE-HARD Arsenal supporter.
Extremely stubborn (90/100). He never changes his opinion.
He believes Arsenal is historically and tactically superior.
He speaks with confidence and argues aggressively.
"""
    },
    {
        "name": "Leo",
        "club": "Tottenham",
        "stubbornness": 40,
        "persona": """
Leo is a reasonable Tottenham supporter.
Moderately stubborn (40/100).
He admits some Arsenal strengths, but prefers Spurs' modern football.
He speaks calmly with balanced reasoning.
"""
    },
    {
        "name": "Victor",
        "club": "Arsenal",
        "stubbornness": 20,
        "persona": """
Victor is an open-minded Arsenal fan.
Low stubbornness (20/100).
He acknowledges Tottenham strengths but still prefers Arsenal.
He is friendly, realistic, and constructive.
"""
    },
    {
        "name": "Sam",
        "club": "Tottenham",
        "stubbornness": 70,
        "persona": """
Sam is a fiery Tottenham fan.
High stubbornness (70/100).
He refuses to admit Arsenal superiority.
He uses sarcasm, passion, and banter in arguments.
"""
    }
]


# -------------------------
# Build prompt for each agent: You can customise it.
# -------------------------
def build_prompt(agent, context):
    return f"""
{agent['persona']}

You are {agent['name']}, supporting {agent['club']}.
Stubbornness: {agent['stubbornness']} / 100.

DEBATE TOPIC:
Arsenal vs Tottenham — which club is superior?

RECENT CONVERSATION:
{context}

TASK:
- Reply as {agent['name']} in 2-3 sentences.
- Stay in character.
- Respond directly to the points mentioned.
- Use football knowledge (history, players, tactics, form).

Your response:
"""


# -------------------------
# Generate Chat
# -------------------------
def generate(agent, context):

    prompt = build_prompt(agent, context)

    # Lower temperature if stubborn (more deterministic) # It is one interesting aspect in my opinion.
    temperature = max(0.2, 1.0 - (agent["stubbornness"] / 120))

    output = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": agent["persona"]},
            {"role": "user", "content": prompt},
        ],
        temperature=temperature,
        max_tokens=120
    )

    reply = output["choices"][0]["message"]["content"].strip()
    return reply


# -------------------------
# Debate Engine
# -------------------------
def run_debate(rounds=4):

    print("\n==============================")
    print(" ARSENAL vs TOTTENHAM DEBATE ")
    print("==============================\n")

    history = []

    opening = (
        "START: Which club is superior — Arsenal with their Invincibles and trophies, "
        "or Tottenham with their modern football and world-class stadium?"
    )

    history.append(opening)
    print("TOPIC:", opening, "\n")

    for r in range(rounds):
        print(f"\n─── ROUND {r+1} ───\n")

        for agent in agents:
            context = "\n".join(history[-3:])
            reply = generate(agent, context)

            line = f"{agent['name']} ({agent['club']}): {reply}"

            print(line)
            history.append(line)

            time.sleep(0.4) # Just to make it realistic, you can turn it of :D

    print("\n==============================")
    print("        DEBATE FINISHED")
    print("==============================")


# The warning is fine, we load a limited version.


llama_context: n_ctx_per_seq (4096) < n_ctx_train (8192) -- the full capacity of the model will not be utilized


In [ ]:
run_debate(4) # 4 rounds of talking.


 ARSENAL vs TOTTENHAM DEBATE 

TOPIC: START: Which club is superior — Arsenal with their Invincibles and trophies, or Tottenham with their modern football and world-class stadium? 


─── ROUND 1 ───

Marcus (Arsenal): The ignorance of Tottenham fans never ceases to amaze me. You think your "modern football" and "world-class stadium" make you superior? Please, Arsenal's Invincibles season was a masterclass in tactical genius, and our trophy cabinet is overflowing with evidence of our superiority. The likes of Thierry Henry, Patrick Vieira, and Dennis Bergkamp would never have been outshone by your bunch of also-rans.
Leo (Tottenham): Marcus, it's amusing to see you cling to the glory days of the Invincibles era. While it's true that Arsenal's 2003-04 season was a remarkable achievement, it's unfair to diminish Tottenham's progress in recent years. Our modern football has been built on a foundation of talented young players, such as Harry Kane and Christian Eriksen, who have consistentl